# Reconnaissance d'expressions faciales — FER2013

**Projet Deep Learning** — Duc Duy et Simon

**Objectif :** entraîner et comprendre un modèle qui reconnaît 7 expressions faciales
(angry, disgust, fear, happy, neutral, sad, surprise) à partir d'images de visages 48×48 en niveaux de gris.

**Dataset :** FER2013 — Kaggle `msambare/fer2013`

**Plan du notebook** (construit progressivement, une cellule par micro-étape) :
1. Exploration et préparation du dataset
2. Baseline dense
3. CNN
4. Entraînement
5. Évaluation et analyse des erreurs
6. Expériences (≥ 3)


### Étape 0 — Configuration commune et téléchargement des données

Cette cellule est lancée **en premier** ; toutes les suivantes en dépendent.

1. **Racine du projet.** En local, c'est le dossier qui contient `.gitignore` : le notebook marche qu'on le lance depuis la racine ou depuis `notebooks/`. Sur Colab, si le dépôt n'a pas été cloné, il n'y a pas de `.gitignore` : on reste alors dans le dossier courant (`/content`).
2. **Constantes partagées.** `DATA_DIR`, `CLASS_NAMES`, `IMAGE_SIZE` et `SEED` sont définies **une seule fois ici** et réutilisées partout : changer une valeur à un endroit la change pour tout le notebook.
3. **Graine aléatoire (*seed*).** `keras.utils.set_random_seed(42)` fixe le hasard de Python, NumPy et TensorFlow (découpe train/validation, poids initiaux, mélange des lots). Deux expériences ne diffèrent alors que par ce qu'on a volontairement changé — indispensable pour comparer nos ≥ 3 expériences. (Sur GPU, quelques opérations restent légèrement non déterministes : de petites variations restent possibles.)
4. **Téléchargement idempotent.** `data/` n'est pas dans Git (≈ 200 Mo). Si `data/fer2013/train` et `data/fer2013/test` existent déjà, la cellule ne télécharge rien ; sinon elle récupère FER2013 avec `kagglehub` (bibliothèque officielle de Kaggle). La relancer 10 fois donne le même résultat qu'une seule fois.

In [ ]:
# Étape 0 : configuration commune + téléchargement de FER2013 s'il manque
import os
import shutil
from pathlib import Path

from tensorflow import keras

# --- 1) Racine du projet ---
# On remonte les dossiers jusqu'à trouver .gitignore (racine du dépôt).
racine = Path.cwd()
while racine != racine.parent and not (racine / ".gitignore").exists():
    racine = racine.parent
if not (racine / ".gitignore").exists():
    # Pas de dépôt (ex. notebook ouvert seul sur Colab) : on reste dans le dossier courant.
    racine = Path.cwd()
os.chdir(racine)  # les cellules suivantes utilisent des chemins relatifs à cette racine
print("Racine du projet :", racine)

# --- 2) Constantes partagées par tout le notebook ---
DATA_DIR = Path("data/fer2013")
# Cet ordre fixe la position du 1 dans le vecteur one-hot :
# 0 = angry, 1 = disgust, 2 = fear, 3 = happy, 4 = neutral, 5 = sad, 6 = surprise.
CLASS_NAMES = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
IMAGE_SIZE = (48, 48)  # taille des images FER2013 (vérifiée à l'étape 1b)
SEED = 42

# --- 3) Graine aléatoire : rend les résultats reproductibles ---
keras.utils.set_random_seed(SEED)

# --- 4) Téléchargement, seulement si les données manquent ---
if (DATA_DIR / "train").is_dir() and (DATA_DIR / "test").is_dir():
    print(f"Données déjà présentes dans {DATA_DIR} — aucun téléchargement.")
else:
    import kagglehub  # déjà installé sur Colab ; en local : pip install kagglehub

    print("Téléchargement du dataset msambare/fer2013 via kagglehub…")
    source = Path(kagglehub.dataset_download("msambare/fer2013"))
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for split in ["train", "test"]:
        if not (DATA_DIR / split).exists():  # jamais réécrit si déjà copié
            shutil.copytree(source / split, DATA_DIR / split)
    print("Dataset copié dans", DATA_DIR.resolve())

## Partie 1 — Recherche, compréhension et préparation des données

### Description du dataset : FER2013

| Élément | Valeur |
|---|---|
| **Source** | Kaggle, [`msambare/fer2013`](https://www.kaggle.com/datasets/msambare/fer2013) (version au format images, un dossier par classe) |
| **Auteurs** | Pierre-Luc Carrier et Aaron Courville. Dataset publié pour le concours *Challenges in Representation Learning* (ICML 2013) : Goodfellow et al., 2013, [arXiv:1307.0414](https://arxiv.org/abs/1307.0414) |
| **Licence** | Database Contents License (DbCL) v1.0, indiquée sur la page Kaggle. Les photos d'origine viennent du web : on utilise le dataset dans un cadre académique |
| **Collecte** | Images trouvées via Google Images avec des mots-clés d'émotions ; visages détectés et recadrés automatiquement, vérifiés par des annotateurs, puis réduits en 48 × 48 niveaux de gris |
| **Nombre d'images** | 35 887 : 28 709 en `train`, 7 178 en `test` |
| **Classes** | 7 : `angry` (colère), `disgust` (dégoût), `fear` (peur), `happy` (joie), `neutral` (neutre), `sad` (tristesse), `surprise` |
| **Dimensions** | 48 × 48 pixels |
| **Format** | Niveaux de gris (1 canal), fichiers `.jpg` |
| **Déséquilibre** | Oui : 436 images `disgust` contre 7 215 `happy` en `train` (≈ 16,5 fois moins) |

**Pourquoi ce dataset ?** Il est public, très utilisé comme référence, et ses visages sont **déjà extraits et recadrés** : c'est exactement l'entrée « visage déjà extrait → CNN → expression » des Parties 1 à 6.

Les chiffres du tableau sont mesurés par les cellules 1a et 1b ci-dessous.

### Étape 1a — Compter les images et visualiser le déséquilibre

Avant de charger les données, on compte les images de chaque dossier : chaque dossier de classe sert d'étiquette (label), c'est cette structure que Keras lira. Le graphique montre le **déséquilibre entre les classes** : un modèle aura tendance à mieux reconnaître les classes fréquentes, c'est pourquoi on regardera le rappel par classe en Partie 5, pas seulement l'accuracy.

In [ ]:
# Micro-étape 1a : compter les images de chaque classe, puis visualiser le déséquilibre
import matplotlib.pyplot as plt

nombres_train = []  # nombre d'images d'entraînement par classe, dans l'ordre de CLASS_NAMES

# Chaque dossier de classe (data/fer2013/train/happy, ...) contient les images de cette classe :
# le nom du dossier EST l'étiquette.
for split in ["train", "test"]:
    total = 0
    for classe in CLASS_NAMES:
        dossier = DATA_DIR / split / classe
        n = len(list(dossier.glob("*.jpg")))  # nombre de fichiers .jpg dans le dossier
        total = total + n
        print(f"{split}/{classe} : {n}")
        if split == "train":
            nombres_train.append(n)
    print(f"{split} total : {total}")
    print()

# --- graphique : une barre par classe (même ordre que CLASS_NAMES) ---
ratio = max(nombres_train) / min(nombres_train)

fig, ax = plt.subplots(figsize=(8, 4))
barres = ax.barh(CLASS_NAMES, nombres_train, height=0.6, color="#2a78d6")
ax.bar_label(barres, padding=4)  # le nombre d'images écrit au bout de chaque barre
ax.invert_yaxis()                # angry en haut, comme dans CLASS_NAMES
ax.set_xlabel("Nombre d'images (train)")
ax.spines[["top", "right"]].set_visible(False)  # axes plus discrets
ax.set_title(f"Images d'entraînement par classe : la plus fréquente en a {ratio:.1f}× plus que la plus rare")
plt.tight_layout()
plt.show()

### Étape 1b — Regarder les images et vérifier le format

**Pourquoi ?** Un modèle apprend à partir de pixels : s'il reçoit des images de tailles ou de modes différents, la couche `Flatten` produira des vecteurs de longueurs différentes et l'entraînement échouera (ou buggera silencieusement).

On fait donc deux choses :
1. **On affiche une grille de 7 classes × 5 images** tirées de `data/fer2013/train/<classe>/` : on vérifie *à l'œil* que les dossiers contiennent bien des visages et que les classes sont cohérentes (utile aussi pour l'oral : montrer des exemples réels).
2. **On vérifie le format sur *toutes* les images** (train + test) avec PIL : taille `(48, 48)`, mode `L` (niveaux de gris) et extension `.jpg`. Un `Counter` (bibliothèque standard Python) compte combien de fois chaque valeur apparaît ; on liste les exceptions plutôt que de les cacher.

In [ ]:
# Micro-étape 1b : exemples d'images + vérification du format (taille, mode, extension)
from collections import Counter

import matplotlib.pyplot as plt
from PIL import Image

N_EXEMPLES = 5  # nombre d'images affichées par classe

# --- 1) grille 7 lignes (classes) x 5 colonnes (exemples) -------------------
fig, axes = plt.subplots(len(CLASS_NAMES), N_EXEMPLES, figsize=(10, 14))
for ligne, classe in enumerate(CLASS_NAMES):
    # on trie les fichiers pour que la grille soit la même à chaque exécution
    fichiers = sorted((DATA_DIR / "train" / classe).glob("*.jpg"))[:N_EXEMPLES]
    for col, chemin in enumerate(fichiers):
        with Image.open(chemin) as img:  # le `with` ferme l'image automatiquement
            axes[ligne, col].imshow(img, cmap="gray")  # images en niveaux de gris
        axes[ligne, col].set_xticks([])  # on masque les graduations : on veut voir l'image
        axes[ligne, col].set_yticks([])
    # nom de la classe écrit à gauche de la ligne
    axes[ligne, 0].set_ylabel(classe, rotation=0, ha="right", va="center")

fig.suptitle("FER2013 — 5 exemples par classe (split train)", fontsize=14)
plt.tight_layout()
plt.show()

# --- 2) vérification du format sur TOUTES les images (train + test) --------
# Counter compte les valeurs rencontrées, ex. {(48, 48): 35887}
tailles, modes, extensions = Counter(), Counter(), Counter()
anomalies = []  # (chemin, description du problème)
n_images = 0

for split in ["train", "test"]:
    for classe in CLASS_NAMES:
        dossier = DATA_DIR / split / classe
        for chemin in sorted(dossier.iterdir()):  # tous les fichiers du dossier de classe
            n_images = n_images + 1
            extension = chemin.suffix.lower()  # ".jpg", ".png", ...
            try:
                with Image.open(chemin) as img:
                    taille = img.size
                    mode = img.mode
            except OSError:  # fichier corrompu ou illisible : on le note et on continue
                anomalies.append((chemin, "illisible"))
                continue
            tailles[taille] += 1
            modes[mode] += 1
            extensions[extension] += 1
            if taille != IMAGE_SIZE or mode != "L" or extension != ".jpg":
                anomalies.append((chemin, f"taille {taille}, mode {mode}, extension {extension}"))

print(f"Images vérifiées : {n_images} (train + test)")
print(f"Tailles observées    : {dict(tailles)}")
print(f"Modes PIL observés   : {dict(modes)}")
print(f"Extensions observées : {dict(extensions)}")
if anomalies:
    print(f"\n{len(anomalies)} exception(s) trouvée(s) sur {n_images} images :")
    for chemin, probleme in anomalies[:20]:  # on n'affiche que les 20 premières
        print(f"  - {chemin} -> {probleme}")
else:
    print(f"\nFormat : 48×48, mode L, .jpg — vérifié sur {n_images} images, aucune exception.")

### Étape 1c — Construire les datasets d'entraînement, de validation et de test

**Pourquoi ?** Jusqu'ici on a seulement *regardé* les fichiers. Le modèle a besoin d'un `tf.data.Dataset` : un flux de lots d'images + d'étiquettes, prêt pour `model.fit()`.

Cinq points à comprendre :

1. **Normalisation** — les pixels sont des entiers entre 0 et 255. On les divise par 255 pour les ramener dans `[0, 1]` : des entrées petites et à la même échelle rendent l'entraînement plus stable et, en général, plus rapide.
2. **Encodage one-hot** — `label_mode="categorical"` transforme l'étiquette `"happy"` en vecteur `[0, 0, 0, 1, 0, 0, 0]` de 7 valeurs (la position du 1 suit l'ordre de `CLASS_NAMES`). C'est la forme attendue par une sortie `softmax` avec la perte `categorical_crossentropy`.
3. **Validation** — 20 % de `data/fer2013/train` est mis de côté (5 741 images) pour mesurer la généralisation pendant l'entraînement et repérer le surapprentissage. Ces images n'entraînent jamais les poids : elles servent seulement à décider quand s'arrêter (`early stopping`, choix du meilleur modèle).
4. **Test** — `data/fer2013/test` (7 178 images) reste intact, sans mélange ni découpe : il est réservé à l'évaluation finale, une seule fois, à la toute fin du projet. Si on l'utilisait pendant le développement, on « tricherait » sans le savoir.

Rappel de l'étape 1b : toutes les images font déjà 48×48 en niveaux de gris, donc la forme d'une image est `(48, 48, 1)` et aucune redimensionnement n'est nécessaire.

5. **Performance** — `.prefetch()` prépare le lot suivant pendant que le modèle calcule sur le lot courant. `.cache()` garde les lots en mémoire après la première lecture : on l'utilise **seulement pour la validation et le test**. Sur l'entraînement, il figerait l'ordre des images de la première époque ; or on veut que les images soient **mélangées différemment à chaque époque**, pour que le modèle ne s'habitue pas à un ordre fixe.

In [ ]:
# Micro-étape 1c : construire les datasets TensorFlow (train / validation / test)
import tensorflow as tf

BATCH_SIZE = 64         # nombre d'images par lot
VALIDATION_SPLIT = 0.2  # 20 % du dossier train est réservé à la validation

# --- 1) train + validation : une seule passe sur data/fer2013/train ---
# subset="both" renvoie directement un tuple (train_ds, val_ds).
# seed=SEED rend la découpe reproductible : la même image tombe toujours du même côté.
# color_mode="grayscale"  -> 1 seul canal, donc une forme (48, 48, 1).
# label_mode="categorical" -> étiquette encodée en one-hot (vecteur de 7 valeurs).
train_ds, val_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR / "train",
    color_mode="grayscale",
    image_size=IMAGE_SIZE,
    label_mode="categorical",
    class_names=CLASS_NAMES,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    subset="both",
    seed=SEED,
)

# --- 2) test : data/fer2013/test, utilisé tel quel, sans découpe ---
# shuffle=False garde l'ordre des fichiers, pour une évaluation stable et reproductible.
test_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR / "test",
    color_mode="grayscale",
    image_size=IMAGE_SIZE,
    label_mode="categorical",
    class_names=CLASS_NAMES,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

# --- 3) nombre d'images réellement présent dans chaque ensemble ---
# .file_paths n'existe que sur le dataset d'origine : on le lit avant le .map ci-dessous.
print("Images en train      :", len(train_ds.file_paths))  # attendu : 22968
print("Images en validation :", len(val_ds.file_paths))    # attendu : 5741
print("Images en test       :", len(test_ds.file_paths))   # attendu : 7178

# --- 4) normalisation : les pixels passent de l'échelle [0, 255] à [0, 1] ---
def normaliser(images, labels):
    return images / 255.0, labels

# .map applique la fonction à chaque lot, au fur et à mesure de la lecture.
# .prefetch prépare le lot suivant pendant que le modèle calcule sur le lot courant.
# .cache garde les lots en mémoire : seulement pour validation et test, car sur train
# il figerait l'ordre des images de la 1re époque (plus de mélange entre les époques).
train_ds = train_ds.map(normaliser).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(normaliser).cache().prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(normaliser).cache().prefetch(tf.data.AUTOTUNE)

# --- 5) contrôle : forme d'un lot et valeurs des pixels ---
images_batch, labels_batch = next(iter(train_ds))  # premier lot : 64 images + étiquettes
print("Forme du lot d'images  :", images_batch.shape)  # attendu : (64, 48, 48, 1)
print("Forme du lot de labels :", labels_batch.shape)  # attendu : (64, 7)
print("Valeur min des pixels  :", float(images_batch.numpy().min()))  # entre 0 et 1
print("Valeur max des pixels  :", float(images_batch.numpy().max()))  # entre 0 et 1

## Partie 2 — Baseline dense

**Pourquoi une baseline ?** Avant de construire un CNN, on entraîne le modèle le plus simple possible. Son score sert de **point de comparaison** : si le CNN ne fait pas nettement mieux, c'est qu'il y a un problème. Sans baseline, impossible de dire si 55 % est un bon ou un mauvais résultat.

**Le modèle : un réseau entièrement connecté (*dense*)**, selon le schéma de l'énoncé `Image → Flatten → Dense + activation → Dense → Softmax`.
1. `Flatten` met les 48 × 48 = **2 304 pixels** de l'image bout à bout, dans un seul vecteur.
2. Une couche cachée `Dense(128, activation="relu")` : chacun des 128 neurones est relié à **toutes** les 2 304 valeurs d'entrée. L'activation `relu` (`max(0, x)`) permet au réseau d'apprendre des relations non linéaires.
3. Une couche de sortie `Dense(7, activation="softmax")` donne **7 probabilités qui somment à 1**, une par expression. La classe prédite est celle qui a la plus grande probabilité.

**Sa limite (attendue) :** après `Flatten`, le réseau ne sait plus quels pixels sont voisins. Un sourire décalé de quelques pixels devient un vecteur complètement différent. C'est exactement ce que le CNN de la Partie 3 corrigera.

**Compilation :**
- `loss="categorical_crossentropy"` : la perte adaptée à des étiquettes one-hot et une sortie softmax (étape 1c). Elle est faible quand le modèle donne une forte probabilité à la bonne classe.
- `optimizer="adam"` : l'algorithme qui ajuste les poids pour faire baisser la perte (une version améliorée de la descente de gradient).
- `metrics=["accuracy"]` : le **taux de bonnes réponses**, plus facile à lire que la perte.

In [ ]:
# Partie 2a : définir et compiler la baseline dense
from tensorflow.keras import layers

NUM_CLASSES = len(CLASS_NAMES)  # 7 expressions

baseline = keras.Sequential(
    [
        keras.Input(shape=IMAGE_SIZE + (1,)),  # (48, 48) + (1,) = (48, 48, 1) : une image, 1 canal
        layers.Flatten(),                       # 48 x 48 x 1 = 2304 valeurs à la suite
        layers.Dense(128, activation="relu"),   # couche cachée : 128 neurones
        layers.Dense(NUM_CLASSES, activation="softmax"),  # 7 probabilités qui somment à 1
    ],
    name="baseline_dense",
)

baseline.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

# summary() affiche chaque couche et son nombre de paramètres (poids à apprendre).
# Exemple : la couche cachée a 2304 x 128 poids + 128 biais = 295 040 paramètres.
baseline.summary()

### Comment le réseau apprend : poids, propagation avant, perte, rétropropagation

**1. L'entrée.** L'image 48 × 48 devient, après `Flatten`, un vecteur $x$ de 2 304 nombres entre 0 et 1 (un par pixel).

**2. Poids et biais.** Chaque neurone calcule une somme pondérée de ses entrées :

$$z = w_1 x_1 + w_2 x_2 + \dots + w_{2304}\, x_{2304} + b$$

Les **poids** $w$ disent quelle importance donner à chaque pixel ; le **biais** $b$ décale le résultat. Ce sont les seuls nombres que le réseau apprend : ici 295 040 + 903 = **295 943 paramètres**.

**3. Propagation avant (*forward pass*).** L'image traverse le réseau couche par couche :
image → `Flatten` → 128 neurones + ReLU → 7 neurones → softmax → 7 probabilités.

**4. Fonctions d'activation.**
- **ReLU** dans la couche cachée : $\text{ReLU}(z) = \max(0, z)$. Sans fonction non linéaire, empiler des couches reviendrait à une seule couche linéaire, incapable d'apprendre des formes complexes.
- **Softmax** en sortie : $\text{softmax}(z_i) = \dfrac{e^{z_i}}{\sum_j e^{z_j}}$ transforme les 7 scores en probabilités positives qui somment à 1. La classe prédite est celle de plus grande probabilité.

**5. La fonction de perte.** La *categorical cross-entropy* vaut $L = -\log(p_{\text{vraie classe}})$ : si le modèle donne 0,72 à la bonne classe, $L \approx 0{,}33$ ; s'il ne lui donne que 0,05, $L \approx 3{,}0$. Plus la perte est basse, meilleur est le modèle.

**6. Rétropropagation (*backpropagation*).** Après chaque lot, on calcule pour **chaque poids** le gradient $\frac{\partial L}{\partial w}$ : de combien la perte change si on modifie un peu ce poids. Le calcul part de la sortie et remonte vers l'entrée, couche par couche, avec la règle de dérivation en chaîne.

**7. Descente de gradient.** Chaque poids est déplacé dans la direction qui fait baisser la perte :

$$w \leftarrow w - \eta \, \frac{\partial L}{\partial w}$$

$\eta$ est le **taux d'apprentissage** (*learning rate*, 0,001 par défaut avec Adam). On fait une mise à jour par lot de 64 images, soit environ 359 mises à jour par époque (22 968 / 64). **Adam** est une variante de la descente de gradient qui adapte automatiquement le pas de chaque poids.

**8. Sortie multiclasse.** 7 classes → 7 neurones de sortie + softmax (pour 2 classes, on aurait 1 neurone + sigmoïde).

### 2b — Entraîner la baseline

`model.fit()` répète l'apprentissage pendant `EPOCHS_BASELINE` **époques** (une époque = le modèle a vu une fois toutes les images d'entraînement, lot par lot).

À la fin de chaque époque, Keras affiche :
- `loss` / `accuracy` : mesurées sur les images d'**entraînement** ;
- `val_loss` / `val_accuracy` : mesurées sur la **validation**, des images que le modèle n'apprend jamais.

Ce qu'on surveille : si `accuracy` continue de monter alors que `val_accuracy` stagne ou baisse, le modèle **apprend par cœur** les images d'entraînement au lieu de généraliser : c'est le **surapprentissage** (*overfitting*). On garde ici un nombre d'époques fixe pour bien voir ce phénomène sur les courbes ; l'arrêt automatique (*early stopping*) viendra en Partie 4.

In [ ]:
# Partie 2b : entraîner la baseline (le jeu de test n'est PAS utilisé ici)
EPOCHS_BASELINE = 20

historique_baseline = baseline.fit(
    train_ds,
    validation_data=val_ds,  # mesuré à chaque époque, sans jamais modifier les poids
    epochs=EPOCHS_BASELINE,
)

### 2c — Courbes d'apprentissage et score de la baseline

On trace la perte et l'accuracy époque par époque, pour l'entraînement et la validation, puis on donne le score final sur la **validation**.

Deux repères pour juger ce score :
- **le hasard** : répondre une classe au hasard donne 1 chance sur 7, soit environ 14 % ;
- **la classe majoritaire** : répondre toujours `happy` (la classe la plus fréquente) donne environ 25 %. Un modèle qui ne dépasse pas ce chiffre n'a rien appris d'utile, même si 25 % paraît mieux que le hasard.

In [ ]:
# Partie 2c : courbes d'apprentissage + score de la baseline sur la validation
import matplotlib.pyplot as plt
import numpy as np

# historique_baseline.history est un dictionnaire : une liste de valeurs par métrique, une valeur par époque
courbes = historique_baseline.history
epoques = range(1, len(courbes["loss"]) + 1)

fig, (ax_perte, ax_accuracy) = plt.subplots(1, 2, figsize=(12, 4))

ax_perte.plot(epoques, courbes["loss"], label="entraînement")
ax_perte.plot(epoques, courbes["val_loss"], label="validation")
ax_perte.set_title("Perte (plus bas = mieux)")
ax_perte.set_xlabel("Époque")
ax_perte.legend()

ax_accuracy.plot(epoques, courbes["accuracy"], label="entraînement")
ax_accuracy.plot(epoques, courbes["val_accuracy"], label="validation")
ax_accuracy.set_title("Accuracy (plus haut = mieux)")
ax_accuracy.set_xlabel("Époque")
ax_accuracy.legend()

fig.suptitle("Baseline dense — courbes d'apprentissage")
plt.tight_layout()
plt.show()

# Score final sur la validation (les poids ne changent pas pendant evaluate)
perte_val, accuracy_val = baseline.evaluate(val_ds, verbose=0)

# Repère "classe majoritaire" : on compte les étiquettes one-hot de la validation
comptes = np.zeros(NUM_CLASSES)
for images, labels in val_ds:
    comptes = comptes + labels.numpy().sum(axis=0)  # somme des vecteurs one-hot = nombre d'images par classe
classe_majoritaire = CLASS_NAMES[int(comptes.argmax())]
accuracy_majoritaire = comptes.max() / comptes.sum()

print(f"Baseline dense — accuracy sur la validation : {accuracy_val:.1%}")
print(f"Repère hasard (1 chance sur {NUM_CLASSES}) : {1 / NUM_CLASSES:.1%}")
print(f"Repère « toujours {classe_majoritaire} » : {accuracy_majoritaire:.1%}")

### 2d — Ce que le réseau renvoie pour une image

Pour une image de validation, on affiche les **7 probabilités** produites par softmax, leur somme (= 1) et la classe prédite (la plus probable), à comparer avec la vraie classe.

In [ ]:
# Partie 2d : les 7 probabilités prédites pour UNE image de validation
images_val, labels_val = next(iter(val_ds))  # un lot de 64 images de validation

# predict attend un lot d'images : images_val[:1] est un lot qui contient seulement la 1re image
probabilites = baseline.predict(images_val[:1], verbose=0)[0]  # 7 probabilités
vraie_classe = CLASS_NAMES[int(labels_val[0].numpy().argmax())]
classe_predite = CLASS_NAMES[int(probabilites.argmax())]

plt.figure(figsize=(2.5, 2.5))
plt.imshow(images_val[0], cmap="gray", vmin=0, vmax=1)  # pixels normalisés entre 0 et 1
plt.title(f"vraie classe : {vraie_classe}")
plt.axis("off")
plt.show()

for classe, probabilite in zip(CLASS_NAMES, probabilites):
    print(f"{classe:10s} {probabilite:.2f}")
print(f"Somme des probabilités : {probabilites.sum():.2f}")
print(f"Classe prédite : {classe_predite} (vraie classe : {vraie_classe})")

## Partie 3 — CNN

**Le modèle : un petit réseau convolutif**, selon le schéma de l'énoncé `Image → Conv2D + ReLU → MaxPooling → Conv2D + ReLU → MaxPooling → Flatten → Dense → Softmax`.
1. `Conv2D(32, 3)` fait glisser 32 **filtres** de 3 × 3 pixels sur l'image. Chaque filtre apprend à détecter un petit motif local (un bord, un coin) **où qu'il soit** dans l'image : c'est ce qui manquait à la baseline dense.
2. `MaxPooling2D(2)` ne garde que la plus grande valeur de chaque carré de 2 × 2 : la carte est deux fois plus petite et la détection supporte mieux un petit décalage du visage.
3. Un second bloc `Conv2D(64, 3)` + `MaxPooling2D(2)` combine ces motifs simples en motifs plus grands (un œil, une bouche).
4. `Flatten` puis `Dense(7, activation="softmax")` transforment ces motifs en **7 probabilités**, comme pour la baseline.

La compilation est identique à celle de la baseline (même perte, même optimiseur, même métrique) : les deux modèles seront donc comparables.

In [ ]:
# Partie 3a : définir et compiler le CNN de base
# On remet la graine à 42 juste avant de créer le modèle : ses poids de départ ne dépendent
# alors pas des cellules lancées avant (la baseline a déjà consommé du hasard).
# Les expériences de la Partie 6 font pareil, ce qui les rend comparables à ce CNN.
keras.utils.set_random_seed(SEED)

cnn = keras.Sequential(
    [
        keras.Input(shape=IMAGE_SIZE + (1,)),          # (48, 48, 1) : une image, 1 canal
        layers.Conv2D(32, 3, activation="relu"),       # 32 filtres 3x3 -> (46, 46, 32)
        layers.MaxPooling2D(2),                        # max sur chaque carré 2x2 -> (23, 23, 32)
        layers.Conv2D(64, 3, activation="relu"),       # 64 filtres 3x3 -> (21, 21, 64)
        layers.MaxPooling2D(2),                        # -> (10, 10, 64)
        layers.Flatten(),                              # 10 x 10 x 64 = 6400 valeurs à la suite
        layers.Dense(NUM_CLASSES, activation="softmax"),  # 7 probabilités qui somment à 1
    ],
    name="cnn_base",
)

cnn.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

# Exemple : la 1re convolution a 3 x 3 x 1 x 32 poids + 32 biais = 320 paramètres,
# contre 295 040 pour la couche cachée de la baseline : les filtres sont partagés sur toute l'image.
cnn.summary()

### Comment un CNN lit une image : filtres, convolution, pooling

**1. Filtre et taille du noyau (*kernel size*).** Un filtre est une petite grille de poids, ici de 3 × 3 (`kernel_size=3`). C'est la plus petite taille qui a un centre et des voisins de tous les côtés ; elle suffit pour repérer un bord ou un coin.

**2. Convolution.** Le filtre glisse sur l'image. À chaque position, on multiplie ses 9 poids par les 9 pixels qu'il recouvre, on additionne le tout et on ajoute un biais. Les **mêmes poids** servent à toutes les positions : le motif est reconnu où qu'il soit, et il y a très peu de paramètres à apprendre.

**3. Carte de caractéristiques (*feature map*).** Le résultat d'un filtre est une nouvelle « image » qui indique où son motif a été trouvé. 32 filtres donnent 32 cartes, d'où la forme `(46, 46, 32)`. Dans la seconde convolution, chaque filtre regarde les 32 cartes à la fois : il a 3 × 3 × 32 poids.

**4. Pas (*stride*).** C'est le déplacement du filtre entre deux positions. Ici 1 pixel pour les convolutions (valeur par défaut) : on ne saute aucune position.

**5. *Padding*.** Avec `padding="valid"` (valeur par défaut), on n'ajoute rien autour de l'image : le filtre reste à l'intérieur, donc la sortie rétrécit de 2 pixels (48 − 3 + 1 = 46). Avec `padding="same"`, on entourerait l'image de zéros pour garder 48. Formule générale :

$$\text{taille de sortie} = \frac{\text{entrée} - \text{noyau} + 2 \times \text{padding}}{\text{pas}} + 1$$

**6. ReLU.** $\max(0, z)$ est appliqué à chaque valeur des cartes : on ne garde que les endroits où le motif est présent, et le réseau devient non linéaire (comme en Partie 2).

**7. *MaxPooling*.** `MaxPooling2D(2)` découpe chaque carte en carrés de 2 × 2 et ne garde que la plus grande valeur : 46 → 23, puis 21 → 10 (21 est impair, la dernière ligne et la dernière colonne sont ignorées). Il n'a aucun paramètre. Il réduit les calculs et rend la détection moins sensible à un décalage d'un pixel.

**8. `Flatten`, `Dense` et sortie.** Les 64 cartes de 10 × 10 sont mises bout à bout (6 400 valeurs), puis `Dense(7)` + softmax donne les 7 probabilités, comme pour la baseline.

**Tableau couche par couche** (à comparer avec `cnn.summary()` ci-dessus) :

| Couche | Forme de sortie | Paramètres | Calcul |
|---|---|---:|---|
| Entrée | (48, 48, 1) | 0 | |
| `Conv2D(32, 3)` + ReLU | (46, 46, 32) | 320 | (3 × 3 × 1 + 1) × 32 |
| `MaxPooling2D(2)` | (23, 23, 32) | 0 | |
| `Conv2D(64, 3)` + ReLU | (21, 21, 64) | 18 496 | (3 × 3 × 32 + 1) × 64 |
| `MaxPooling2D(2)` | (10, 10, 64) | 0 | |
| `Flatten` | (6400,) | 0 | 10 × 10 × 64 |
| `Dense(7)` + softmax | (7,) | 44 807 | 6 400 × 7 + 7 |
| **Total** | | **63 623** | |

**Pourquoi cette architecture ?**
- C'est la structure proposée par l'énoncé : un point de départ simple, que l'on peut expliquer couche par couche.
- Deux blocs suffisent pour des images de 48 × 48 : après deux *poolings*, les cartes ne font plus que 10 × 10.
- On double le nombre de filtres (32 → 64) quand la taille des cartes diminue : les motifs deviennent plus variés à mesure qu'ils sont plus grands.
- Le CNN a environ **4,6 fois moins de paramètres** que la baseline dense (63 623 contre 295 943), tout en tenant compte du voisinage des pixels.
- Les variantes (plus de filtres, couche `Dense` cachée, *dropout*…) seront testées une par une en Partie 6.

## Partie 4 — Entraînement du CNN

### Les choix d'entraînement et leur justification

| Choix | Valeur | Pourquoi |
|---|---|---|
| Fonction de perte | `categorical_crossentropy` | Chaque image a une seule classe parmi 7, codée en one-hot (étape 1c), et la sortie est un softmax. La perte vaut $-\log(p_{\text{vraie classe}})$ : elle est petite quand le modèle donne une forte probabilité à la bonne classe et devient très grande quand il se trompe avec assurance. |
| Optimiseur | Adam, taux d'apprentissage 0,001 (valeur par défaut de Keras) | Adam est une descente de gradient qui adapte le pas de chaque poids. Il fonctionne bien sans réglage sur un petit réseau comme le nôtre. On garde le taux par défaut car la perte de validation baisse dès les premières époques (courbes 4b) : rien n'indique un pas trop grand ou trop petit. |
| Taille de lot (*batch size*) | 64 | Une mise à jour des poids tous les 64 exemples, soit 359 mises à jour par époque. Avec des lots plus petits, chaque époque est plus longue et le gradient plus bruité ; avec des lots plus grands, on fait moins de mises à jour par époque. 64 est une valeur courante et 64 images de 48 × 48 tiennent sans problème en mémoire. |
| Nombre d'époques | 30 au maximum, arrêt automatique (patience 3) | On ne connaît pas le bon nombre d'époques à l'avance. On met un plafond large et c'est `val_loss` qui décide quand s'arrêter (détails en 4a). |
| Métriques | accuracy pendant l'entraînement, puis rappel par classe en Partie 5 | L'accuracy est simple à lire et se compare directement à la baseline. Elle ne suffit pas avec des classes déséquilibrées : un modèle peut avoir une accuracy correcte en ne reconnaissant presque jamais `disgust`. C'est pour cela qu'on regarde aussi la matrice de confusion et le rappel de chaque classe. |

On surveille `val_loss` plutôt que `val_accuracy` pour l'arrêt automatique : la perte tient compte de la confiance du modèle et bouge plus finement d'une époque à l'autre, alors que l'accuracy ne change que quand une prédiction bascule.

### 4a — Entraîner le CNN avec arrêt automatique

On entraîne le CNN comme la baseline, avec une différence : l'**arrêt automatique** (*early stopping*). À la fin de chaque époque, Keras regarde `val_loss`. Si elle ne s'est pas améliorée depuis 3 époques (`patience=3`), l'entraînement s'arrête et on reprend les poids de la **meilleure époque** (`restore_best_weights=True`).

**Pourquoi ?** Passé un certain point, le modèle apprend par cœur les images d'entraînement : `loss` continue de baisser mais `val_loss` remonte. Le modèle qu'on veut garder est celui d'avant ce point. `EPOCHS_CNN` n'est donc qu'un maximum.

In [ ]:
# Partie 4a : entraîner le CNN (le jeu de test n'est PAS utilisé ici)
EPOCHS_CNN = 30  # maximum : l'arrêt automatique stoppe en général bien avant

# Si val_loss ne s'améliore plus pendant 3 époques, on arrête
# et on reprend les poids de la meilleure époque (avant le surapprentissage).
arret_automatique = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

historique_cnn = cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_CNN,
    callbacks=[arret_automatique],
)

perte_val_cnn, accuracy_val_cnn = cnn.evaluate(val_ds, verbose=0)
print(f"CNN — accuracy sur la validation            : {accuracy_val_cnn:.1%}")
print(f"Baseline dense — accuracy sur la validation : {accuracy_val:.1%}")

### 4b — Courbes d'apprentissage du CNN

On trace les mêmes courbes que pour la baseline (2c). La ligne en pointillés marque la meilleure époque, celle dont l'arrêt automatique a gardé les poids. Le graphique est mis dans une fonction `tracer_courbes`, réutilisée en Partie 6.

In [ ]:
# Partie 4b : courbes d'apprentissage du CNN (même graphique qu'en 2c)
def tracer_courbes(historique, titre):
    courbes = historique.history
    epoques = range(1, len(courbes["loss"]) + 1)
    # meilleure époque = plus petite val_loss : ce sont ses poids que l'arrêt automatique a gardés
    meilleure_epoque = int(np.argmin(courbes["val_loss"])) + 1

    fig, (ax_perte, ax_accuracy) = plt.subplots(1, 2, figsize=(12, 4))
    ax_perte.plot(epoques, courbes["loss"], label="entraînement")
    ax_perte.plot(epoques, courbes["val_loss"], label="validation")
    ax_perte.set_title("Perte (plus bas = mieux)")
    ax_accuracy.plot(epoques, courbes["accuracy"], label="entraînement")
    ax_accuracy.plot(epoques, courbes["val_accuracy"], label="validation")
    ax_accuracy.set_title("Accuracy (plus haut = mieux)")
    for ax in (ax_perte, ax_accuracy):
        ax.axvline(meilleure_epoque, color="gray", linestyle="--", label=f"meilleure époque ({meilleure_epoque})")
        ax.set_xlabel("Époque")
        ax.legend()
    fig.suptitle(titre)
    plt.tight_layout()
    plt.show()


tracer_courbes(historique_cnn, "CNN — courbes d'apprentissage")

# Écart entre entraînement et validation : s'il grandit, le modèle apprend par cœur.
courbes = historique_cnn.history
meilleure = int(np.argmin(courbes["val_loss"]))  # position dans la liste (0 = 1re époque)
print(f"Époques effectuées : {len(courbes['loss'])}, meilleure époque : {meilleure + 1}")
print(f"Meilleure époque : accuracy train {courbes['accuracy'][meilleure]:.1%}, "
      f"validation {courbes['val_accuracy'][meilleure]:.1%}, val_loss {courbes['val_loss'][meilleure]:.3f}")
print(f"Dernière époque  : accuracy train {courbes['accuracy'][-1]:.1%}, "
      f"validation {courbes['val_accuracy'][-1]:.1%}, val_loss {courbes['val_loss'][-1]:.3f}")

**Lecture des courbes.** Pendant les premières époques, les deux pertes baissent ensemble : ce que le modèle apprend sert aussi sur des images qu'il n'a jamais vues. Ensuite elles se séparent. La perte d'entraînement continue de descendre, alors que la perte de validation ne baisse presque plus, atteint son minimum (ligne en pointillés) puis remonte. Pour l'accuracy, l'entraînement continue de monter pendant que la validation plafonne autour de 50 %.

C'est du surapprentissage : après la meilleure époque, le modèle ne progresse plus que sur les images d'entraînement, qu'il commence à apprendre par cœur. L'arrêt automatique stoppe l'entraînement 3 époques après ce minimum et remet les poids de la meilleure époque. La cellule précédente affiche la meilleure époque et l'écart entre entraînement et validation pour cette exécution. Réduire cet écart est l'objet de plusieurs expériences de la Partie 6 (*dropout*, augmentation de données).

## Partie 5 — Évaluation du CNN

Toute cette partie utilise la **validation**. Le jeu de test reste réservé à l'évaluation finale du modèle retenu, une seule fois.

### 5a — Matrice de confusion et rappel par classe

L'accuracy donne un seul chiffre. Comme les classes sont déséquilibrées, un modèle peut avoir une accuracy correcte en ratant presque toujours `disgust`. La **matrice de confusion** montre le détail :
- chaque **ligne** est une vraie classe, chaque **colonne** une classe prédite ;
- la **diagonale** compte les bonnes réponses ; toutes les autres cases sont des erreurs, et on voit **avec quelle classe** le modèle confond.

Le **rappel** (*recall*) d'une classe est la part de ses images que le modèle a retrouvées : case de la diagonale ÷ total de la ligne.

In [ ]:
# Partie 5a : matrice de confusion du CNN sur la validation
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

# On parcourt la validation une seule fois : pour chaque lot, on garde les vraies
# étiquettes et les probabilités prédites, ainsi que les images (pour les exemples de 5d).
# Les trois listes restent ainsi dans le même ordre.
images_lots = []
labels_lots = []
probabilites_lots = []
for images, labels in val_ds:
    images_lots.append(images.numpy())
    labels_lots.append(labels.numpy())
    probabilites_lots.append(cnn.predict_on_batch(images))
labels_onehot = np.concatenate(labels_lots)            # (5741, 7) : étiquettes one-hot
probabilites_cnn = np.concatenate(probabilites_lots)   # (5741, 7) : probabilités du CNN

vraies_classes = labels_onehot.argmax(axis=1)       # position du 1 = numéro de la vraie classe
classes_predites = probabilites_cnn.argmax(axis=1)  # classe la plus probable selon le CNN

matrice = confusion_matrix(vraies_classes, classes_predites)
ConfusionMatrixDisplay(matrice, display_labels=CLASS_NAMES).plot(cmap="Blues", xticks_rotation=45)
plt.title("CNN — matrice de confusion (validation)")
plt.show()

# Rappel par classe : bonnes réponses de la ligne / nombre d'images de cette classe
for numero, classe in enumerate(CLASS_NAMES):
    total_classe = matrice[numero].sum()
    print(f"{classe:10s} rappel = {matrice[numero, numero] / total_classe:.1%}  ({matrice[numero, numero]} / {total_classe})")

### 5b — Courbes ROC (une classe contre toutes les autres)

Une courbe ROC se trace pour une question à deux réponses. Avec 7 classes, on en trace donc 7 : pour chacune, « l'image est-elle de cette classe, **oui ou non** ? » (*one-vs-rest*).

Pour une classe, par exemple `happy`, on décide « oui » quand la probabilité prédite dépasse un **seuil**. En faisant varier ce seuil de 1 à 0, on obtient pour chaque valeur :
- le **taux de vrais positifs** (axe vertical) : part des vraies images `happy` reconnues — c'est le rappel ;
- le **taux de faux positifs** (axe horizontal) : part des autres images prises à tort pour `happy`.

Plus la courbe monte vite vers le coin en haut à gauche, mieux la classe est séparée des autres. L'**AUC** (aire sous la courbe) résume la courbe en un nombre : 1,0 = séparation parfaite, 0,5 = hasard (la diagonale en pointillés).

**Attention :** l'AUC mesure si le modèle donne des probabilités plus élevées aux bonnes images, pas s'il prédit finalement la bonne classe. Une classe rare comme `disgust` peut avoir une AUC élevée et un rappel faible : il faut lire les courbes ROC **avec** la matrice de confusion.

In [ ]:
# Partie 5b : courbes ROC du CNN sur la validation, une courbe par classe
from sklearn.metrics import auc, roc_curve

plt.figure(figsize=(7, 6))
for numero, classe in enumerate(CLASS_NAMES):
    # labels_onehot[:, numero] vaut 1 si l'image est de cette classe, 0 sinon ;
    # probabilites_cnn[:, numero] est la probabilité que le CNN donne à cette classe.
    taux_faux_positifs, taux_vrais_positifs, seuils = roc_curve(
        labels_onehot[:, numero], probabilites_cnn[:, numero]
    )
    aire = auc(taux_faux_positifs, taux_vrais_positifs)
    plt.plot(taux_faux_positifs, taux_vrais_positifs, label=f"{classe} (AUC = {aire:.2f})")

plt.plot([0, 1], [0, 1], "k--", label="hasard (AUC = 0.50)")
plt.xlabel("Taux de faux positifs")
plt.ylabel("Taux de vrais positifs (rappel)")
plt.title("CNN — courbes ROC, une classe contre les autres (validation)")
plt.legend()
plt.show()

### 5c — Classes les plus confondues et classes les plus difficiles

On lit la matrice de confusion hors de la diagonale. Le code liste toutes les cases d'erreur (vraie classe → classe prédite) et les trie de la plus grande à la plus petite. Chaque nombre est aussi donné en pourcentage de sa vraie classe, car les classes n'ont pas la même taille : 50 erreurs pèsent beaucoup plus pour `disgust` (73 images) que pour `happy`.

On classe ensuite les 7 classes de la plus difficile à la plus facile, selon leur rappel.

In [ ]:
# Partie 5c : confusions les plus fréquentes et classes les plus difficiles (validation)
confusions = []  # (nombre d'erreurs, vraie classe, classe prédite)
for vraie in range(NUM_CLASSES):
    for predite in range(NUM_CLASSES):
        if vraie != predite:  # hors diagonale = erreurs
            confusions.append((matrice[vraie, predite], vraie, predite))
confusions.sort(reverse=True)  # du plus grand nombre d'erreurs au plus petit

print("Les 5 confusions les plus fréquentes (vraie classe -> classe prédite) :")
for nombre, vraie, predite in confusions[:5]:
    part = nombre / matrice[vraie].sum()
    print(f"  {CLASS_NAMES[vraie]:8s} -> {CLASS_NAMES[predite]:8s} : {nombre} images "
          f"({part:.1%} des images {CLASS_NAMES[vraie]})")

print("\nClasses de la plus difficile à la plus facile :")
rappels = matrice.diagonal() / matrice.sum(axis=1)  # bonnes réponses / images de la classe
for numero in np.argsort(rappels):  # argsort : numéros des classes, du plus petit rappel au plus grand
    print(f"  {CLASS_NAMES[numero]:8s} rappel = {rappels[numero]:.1%}")

**Ce que montrent la matrice et ces chiffres.** Les valeurs exactes sont affichées par la cellule précédente ; elles varient un peu d'une exécution à l'autre, mais les tendances ci-dessous se retrouvent à chaque fois.

`happy` et `surprise` sont les classes les mieux reconnues. Leurs signes sont très visibles, même en 48 × 48 : la bouche qui sourit pour l'une, la bouche ouverte et les yeux écarquillés pour l'autre. `happy` est aussi la classe qui a le plus d'images d'entraînement.

Les confusions les plus fréquentes font presque toutes intervenir `sad` et `neutral` : `neutral → sad`, `fear → sad`, `sad → neutral`, `angry → sad`. La colonne `sad` de la matrice reçoit beaucoup d'images des autres classes. Un visage triste, neutre ou inquiet a souvent la bouche fermée et peu de mouvement : la différence tient à quelques pixels autour des sourcils et des coins de la bouche.

Les classes les plus difficiles :
- `disgust`. C'est d'abord un problème de quantité : 436 images d'entraînement contre 7 215 pour `happy`. Le modèle voit rarement cette classe et se tromper sur elle coûte peu à l'accuracy globale. Quand il se trompe, il répond souvent `angry` : dans les deux expressions, les sourcils se froncent et le nez se plisse.
- `fear`. Elle n'a pas de signe qui lui soit propre : les yeux grands ouverts et la bouche ouverte la font prendre pour `surprise`, une bouche crispée pour `sad`. Ses erreurs sont réparties sur toutes les autres classes.
- `angry`, souvent prise pour `sad` ou `neutral`, probablement parce que des sourcils froncés se voient mal à cette résolution.

Une autre cause probable, que nous n'avons pas mesurée, est le bruit dans les étiquettes. Les images de FER2013 viennent de recherches sur le web et certaines expressions sont ambiguës même pour un humain : une partie des « erreurs » du modèle n'en sont peut-être pas (voir 5d).

### 5d — Exemples : image, vraie classe, classe prédite, probabilité

On affiche 4 images bien classées (en vert) et 8 erreurs (en rouge). Pour les erreurs, on prend celles où le CNN était **le plus sûr de lui** : ce sont les plus instructives, puisque le modèle s'y trompe avec une forte probabilité. Sous chaque image : la vraie classe avec la probabilité que le modèle lui a donnée, puis la classe prédite avec sa probabilité.

In [ ]:
# Partie 5d : quelques bonnes prédictions et plusieurs erreurs, sur la validation
images_val_toutes = np.concatenate(images_lots)  # (5741, 48, 48, 1), même ordre que vraies_classes

bonnes = np.where(classes_predites == vraies_classes)[0]   # numéros des images bien classées
fausses = np.where(classes_predites != vraies_classes)[0]  # numéros des images mal classées

# Erreurs triées par probabilité de la classe prédite, de la plus forte à la plus faible
confiance_fausses = probabilites_cnn[fausses].max(axis=1)
fausses_choisies = fausses[np.argsort(confiance_fausses)[::-1][:8]]
exemples = list(bonnes[:4]) + list(fausses_choisies)

fig, axes = plt.subplots(2, 6, figsize=(14, 6))
for ax, numero in zip(axes.flat, exemples):
    vraie = vraies_classes[numero]
    predite = classes_predites[numero]
    couleur = "green" if vraie == predite else "red"
    ax.imshow(images_val_toutes[numero], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"vraie : {CLASS_NAMES[vraie]} ({probabilites_cnn[numero, vraie]:.0%})\n"
                 f"prédite : {CLASS_NAMES[predite]} ({probabilites_cnn[numero, predite]:.0%})",
                 color=couleur, fontsize=10)
    ax.axis("off")
fig.suptitle("CNN — 4 bonnes prédictions (vert) et les 8 erreurs les plus confiantes (rouge), validation")
plt.tight_layout()
plt.show()

**Analyse des erreurs.** Pour les erreurs affichées, le modèle donne presque toute la probabilité à la mauvaise classe et presque rien à la vraie. Les images exactes changent un peu d'une exécution à l'autre, mais on retrouve toujours deux sortes d'erreurs.

Certaines viennent d'expressions qui se ressemblent. Des images `fear` sont prédites `surprise` : yeux écarquillés et bouche grande ouverte, parfois avec les mains sur les joues, c'est-à-dire exactement ce que le modèle associe à la surprise. Une bouche qui crie, dents visibles, fait prendre `fear` pour `angry`. Une tête appuyée sur la main, les yeux baissés, fait prendre `neutral` pour `sad`.

D'autres semblent venir d'étiquettes fausses. Des images étiquetées `surprise`, `sad` ou `neutral` montrent des personnes qui sourient ou rient et sont prédites `happy` : à l'œil, la prédiction paraît plus juste que l'étiquette. Une grimace qui montre les dents peut aussi être confondue avec un sourire.

Le modèle semble donc s'appuyer surtout sur la forme de la bouche et l'ouverture des yeux, et moins sur des indices plus fins comme la position des sourcils. Les étiquettes douteuses, elles, limitent probablement le score que n'importe quel modèle peut atteindre sur FER2013.

## Partie 6 — Expériences

Le CNN de la Partie 4 (`cnn_base`) sert de référence. Chaque expérience part du **dernier modèle retenu** et ne change qu'une chose, pour pouvoir attribuer l'effet observé à ce changement. Tout le reste est identique :

- même découpage train / validation, même graine (remise à 42 avant de créer chaque modèle) ;
- même compilation (Adam, taux 0,001, `categorical_crossentropy`), mêmes lots de 64 ;
- au plus 30 époques avec le même arrêt automatique sur `val_loss` (patience 3, meilleurs poids gardés) ;
- tous les scores sont mesurés sur la **validation**. Le jeu de test n'est utilisé qu'une fois, à la fin, pour le modèle choisi.

Pour chaque modèle on note l'accuracy et la perte de validation, la meilleure époque, l'écart d'accuracy entre entraînement et validation à cette époque (signe de surapprentissage) et le rappel de `disgust`, la classe la plus rare.

| Expérience | Changement | Hypothèse |
|---|---|---|
| E1 | une couche `Dense(128)` + ReLU avant la sortie (schéma de la figure 2 de l'énoncé) | une couche cachée combine les motifs trouvés par les convolutions avant de décider |
| E2 | un `Dropout(0.5)` juste avant cette couche `Dense(128)` | le modèle surapprend dès les premières époques (4b) ; le *dropout* devrait réduire l'écart entre entraînement et validation |
| E3 | un troisième bloc `Conv2D(128)` + `MaxPooling` | des motifs plus grands, et beaucoup moins de valeurs à l'entrée de la couche dense |
| E4 | de l'augmentation de données (miroir horizontal, petite rotation, petit zoom) | le modèle voit des variantes des visages à chaque époque et apprend moins par cœur |
| E5 | une pondération des classes (`class_weight`) | une erreur sur `disgust` coûte plus cher, son rappel devrait monter, sans doute au prix d'un peu d'accuracy |

Les critères de choix sont fixés avant de regarder les résultats : on garde un changement s'il améliore l'accuracy de validation ; à accuracy presque égale (moins d'un point d'écart), on regarde la perte de validation et l'écart entre entraînement et validation.

### 6a — Deux petites fonctions pour ne pas répéter le code

- `entrainer(modele, poids_classes=None)` compile et entraîne un modèle **exactement** comme le CNN de la Partie 4.
- `mesurer(nom, changement, modele, historique)` calcule les scores de validation et ajoute une ligne à la liste `resultats`, qui deviendra le tableau comparatif.

On mesure tout de suite `cnn_base`, déjà entraîné en 4a : c'est la première ligne du tableau.

In [ ]:
# Partie 6a : fonctions communes aux expériences (le jeu de test n'est PAS utilisé ici)
resultats = []  # une ligne du tableau comparatif par modèle
NUMERO_DISGUST = CLASS_NAMES.index("disgust")


def entrainer(modele, poids_classes=None):
    modele.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return modele.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_CNN,
        callbacks=[arret_automatique],  # même arrêt automatique qu'en 4a
        class_weight=poids_classes,     # None : toutes les classes comptent autant
        verbose=2,                      # une ligne par époque, pour garder le notebook lisible
    )


def mesurer(nom, changement, modele, historique):
    perte, accuracy = modele.evaluate(val_ds, verbose=0)
    # val_ds est en cache : ses images sortent toujours dans le même ordre,
    # donc ces prédictions sont alignées avec vraies_classes (calculé en 5a).
    predites = modele.predict(val_ds, verbose=0).argmax(axis=1)
    images_disgust = vraies_classes == NUMERO_DISGUST
    rappel_disgust = (predites[images_disgust] == NUMERO_DISGUST).mean()

    courbes = historique.history
    meilleure = int(np.argmin(courbes["val_loss"]))  # époque dont les poids ont été gardés
    ecart = courbes["accuracy"][meilleure] - courbes["val_accuracy"][meilleure]

    resultats.append({
        "nom": nom,
        "changement": changement,
        "accuracy": accuracy,
        "perte": perte,
        "meilleure_epoque": meilleure + 1,
        "epoques": len(courbes["loss"]),
        "ecart": ecart,
        "rappel_disgust": rappel_disgust,
    })
    print(f"{nom} : accuracy {accuracy:.1%}, perte {perte:.3f}, meilleure époque {meilleure + 1} "
          f"sur {len(courbes['loss'])}, écart train-val {ecart:+.1%}, rappel disgust {rappel_disgust:.1%}")


mesurer("cnn_base", "CNN de la Partie 3 (référence)", cnn, historique_cnn)

### 6b — Expérience 1 : une couche `Dense(128)` cachée

**Hypothèse.** Dans `cnn_base`, les 6 400 valeurs sorties des convolutions vont directement vers les 7 sorties. Une couche cachée de 128 neurones peut d'abord combiner ces motifs (par exemple des coins de bouche relevés avec des yeux plissés) avant de décider. C'est aussi la structure de la figure 2 de l'énoncé.

**Changement.** `Flatten → Dense(128, relu) → Dense(7, softmax)` au lieu de `Flatten → Dense(7, softmax)`. Le nombre de paramètres passe de 63 623 à 839 047, presque tous dans la nouvelle couche (6 400 × 128 + 128 = 819 328).

In [ ]:
# Expérience 1 : cnn_base + une couche Dense(128) cachée
keras.utils.set_random_seed(SEED)  # même point de départ du hasard pour chaque modèle

modele_e1 = keras.Sequential(
    [
        keras.Input(shape=IMAGE_SIZE + (1,)),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),  # NOUVEAU : couche cachée
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ],
    name="e1_dense128",
)
historique_e1 = entrainer(modele_e1)
mesurer("e1_dense128", "+ Dense(128) cachée", modele_e1, historique_e1)

**Résultat E1.** L'accuracy reste très proche de celle de `cnn_base` : l'écart est de l'ordre d'un point, ce qui peut venir du simple hasard de l'initialisation. La perte de validation est un peu meilleure. Le modèle surapprend plus tôt : sa meilleure époque arrive avant celle de `cnn_base` et l'écart entre entraînement et validation grandit. Ce n'est pas surprenant avec 13 fois plus de paramètres pour le même nombre d'images.

**Décision : on garde E1** comme base de la suite. L'accuracy est presque la même et la perte de validation un peu meilleure ; c'est aussi la structure du schéma de l'énoncé. Son surapprentissage rapide est justement ce que E2 cherche à corriger.

### 6c — Expérience 2 : *Dropout*

**Hypothèse.** Avec E1, la meilleure époque arrive tôt ; ensuite la perte de validation remonte alors que l'accuracy d'entraînement continue de grimper. Presque tous les paramètres sont dans la couche `Dense(128)` (819 328 sur 839 047). Le *dropout* devrait freiner l'apprentissage par cœur dans cette couche et repousser la meilleure époque.

**Changement.** Un `Dropout(0.5)` entre `Flatten` et `Dense(128)`. Pendant l'entraînement, il met à zéro au hasard la moitié des 6 400 valeurs, différemment à chaque lot : le réseau ne peut plus compter sur quelques valeurs précises et doit répartir l'information. À la prédiction, le *dropout* est désactivé et toutes les valeurs sont utilisées. Il n'ajoute aucun paramètre.

On l'a placé avant `Dense(128)` parce que c'est là que se trouvent presque tous les paramètres du modèle, donc l'essentiel du risque d'apprendre par cœur.

In [ ]:
# Expérience 2 : modèle E1 + Dropout(0.5) avant la couche Dense(128)
keras.utils.set_random_seed(SEED)

modele_e2 = keras.Sequential(
    [
        keras.Input(shape=IMAGE_SIZE + (1,)),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dropout(0.5),  # NOUVEAU : éteint au hasard 50 % des valeurs, seulement à l'entraînement
        layers.Dense(128, activation="relu"),
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ],
    name="e2_dropout",
)
historique_e2 = entrainer(modele_e2)
mesurer("e2_dropout", "+ Dropout(0.5) avant Dense(128)", modele_e2, historique_e2)

**Résultat E2.** L'accuracy et la perte de validation sont au moins aussi bonnes qu'avec E1. Le *dropout* freine l'apprentissage par cœur : selon les exécutions, il réduit l'écart entre entraînement et validation ou retarde la meilleure époque. Il retarde le surapprentissage sans le supprimer. Le rappel de `disgust` progresse un peu.

**Décision : on garde le *dropout*.**

### 6d — Expérience 3 : un troisième bloc de convolution

**Hypothèse.** Avec deux blocs, chaque valeur de la dernière carte ne « voit » qu'un morceau du visage. Un troisième bloc combine ces morceaux en motifs plus grands (la bouche avec les joues, les sourcils avec les yeux). Il réduit aussi la taille des cartes avant `Flatten` : la couche `Dense(128)` reçoit 2 048 valeurs au lieu de 6 400, donc le modèle a moins de poids à apprendre par cœur.

**Changement.** Un bloc `Conv2D(128, 3)` + `MaxPooling2D(2)` après le deuxième bloc. On double encore le nombre de filtres (32 → 64 → 128), comme en Partie 3.

| Couche | Forme de sortie | Paramètres |
|---|---|---:|
| après le 2e bloc | (10, 10, 64) | |
| `Conv2D(128, 3)` + ReLU | (8, 8, 128) | (3 × 3 × 64 + 1) × 128 = 73 856 |
| `MaxPooling2D(2)` | (4, 4, 128) | 0 |
| `Flatten` | (2048,) | 0 |
| `Dense(128)` | (128,) | 2 048 × 128 + 128 = 262 272 |
| **Total du modèle** | | **355 847** (contre 839 047 pour E2) |

In [ ]:
# Expérience 3 : modèle E2 + un troisième bloc de convolution
keras.utils.set_random_seed(SEED)

modele_e3 = keras.Sequential(
    [
        keras.Input(shape=IMAGE_SIZE + (1,)),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(128, 3, activation="relu"),  # NOUVEAU : 3e bloc -> (8, 8, 128)
        layers.MaxPooling2D(2),                     # NOUVEAU : -> (4, 4, 128)
        layers.Flatten(),                           # 4 x 4 x 128 = 2048 valeurs
        layers.Dropout(0.5),
        layers.Dense(128, activation="relu"),
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ],
    name="e3_3_blocs",
)
historique_e3 = entrainer(modele_e3)
mesurer("e3_3_blocs", "+ 3e bloc Conv2D(128) + MaxPooling", modele_e3, historique_e3)

**Résultat E3.** C'est le plus gros gain des expériences : plusieurs points d'accuracy en plus par rapport à E2 et la plus petite perte de validation, avec moins de la moitié des paramètres de E2. Le modèle progresse plus longtemps avant de surapprendre et le rappel de `disgust` augmente nettement. Le surapprentissage n'a pas disparu : à la meilleure époque, l'accuracy d'entraînement reste au-dessus de celle de validation.

**Décision : on garde le troisième bloc.** Ici, des convolutions plus profondes ont plus aidé que la grande couche dense de E1.

### 6e — Expérience 4 : augmentation de données

**Hypothèse.** E3 surapprend encore : à sa meilleure époque, l'accuracy d'entraînement dépasse celle de validation. Avec l'augmentation, le modèle ne voit jamais deux fois exactement la même image, ce qui devrait réduire cet écart et, peut-être, améliorer la validation.

**Changement.** Trois couches Keras au début du modèle, qui transforment chaque image au hasard, différemment à chaque époque :
- `RandomFlip("horizontal")` : miroir gauche-droite. Une expression reste la même dans un miroir. On ne retourne pas haut-bas, car il n'y a pas de visages à l'envers dans les données ;
- `RandomRotation(0.05)` : rotation d'au plus 0,05 tour, soit ± 18°, comme une tête un peu penchée ;
- `RandomZoom(0.1)` : zoom avant ou arrière d'au plus 10 %, comme un visage recadré un peu plus large ou plus serré.

Ces couches ne sont **actives que pendant l'entraînement** : en validation, en test et pour la démonstration, l'image passe sans modification. Le nombre d'images par époque ne change pas, ce sont leurs variantes qui changent d'une époque à l'autre.

In [ ]:
# Expérience 4 : modèle E3 + augmentation de données
keras.utils.set_random_seed(SEED)

modele_e4 = keras.Sequential(
    [
        keras.Input(shape=IMAGE_SIZE + (1,)),
        # NOUVEAU : variantes aléatoires des images, actives seulement pendant l'entraînement
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(128, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(128, activation="relu"),
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ],
    name="e4_augmentation",
)
historique_e4 = entrainer(modele_e4)
mesurer("e4_augmentation", "+ augmentation (miroir, rotation, zoom)", modele_e4, historique_e4)

**Résultat E4.** L'écart entre entraînement et validation devient presque nul, voire négatif : le modèle ne surapprend plus, il fait même parfois mieux sur la validation que sur les images augmentées de l'entraînement. En contrepartie, il apprend beaucoup plus lentement, et quand l'arrêt automatique le stoppe, son accuracy de validation reste en dessous de celle de E3. Le rappel de `disgust` baisse. Avec plus d'époques ou une patience plus grande, l'augmentation ferait peut-être mieux, mais nous ne l'avons pas mesuré : changer la patience serait une autre expérience.

**Décision : on ne garde pas l'augmentation.** Avec le même protocole que les autres modèles (30 époques au plus, patience 3), elle fait perdre de l'accuracy. E5 repart donc de E3.

### 6f — Expérience 5 : pondération des classes

**Hypothèse.** Même dans E3, le meilleur modèle jusqu'ici, `disgust` reste la classe la moins bien reconnue. Le modèle voit cette classe 16 fois moins souvent que `happy`, donc ses erreurs sur `disgust` pèsent peu dans la perte. En donnant plus de poids aux images des classes rares, son rappel devrait monter, sans doute au prix d'un peu d'accuracy sur les classes fréquentes.

**Changement.** Même modèle que E3, le dernier retenu (l'augmentation de E4 n'a pas été gardée), entraîné avec `class_weight`. Le poids d'une classe vaut $\dfrac{\text{nombre total d'images}}{7 \times \text{nombre d'images de la classe}}$ : une classe de taille moyenne a un poids proche de 1, une classe rare un poids plus grand. Dans la perte, une erreur sur une image est multipliée par le poids de sa classe.

On calcule ces poids avec les comptes de l'étape 1a (`nombres_train`, dossier `train` complet). La validation est tirée au hasard dans ce dossier, donc les proportions sont presque les mêmes que dans la partie entraînement.

In [ ]:
# Expérience 5 : modèle E3 (le dernier retenu) entraîné avec un poids par classe
total_images = sum(nombres_train)
poids_classes = {}
for numero, nombre in enumerate(nombres_train):
    poids_classes[numero] = total_images / (NUM_CLASSES * nombre)  # classe rare -> poids élevé
    print(f"{CLASS_NAMES[numero]:8s} {nombre:5d} images -> poids {poids_classes[numero]:.2f}")

keras.utils.set_random_seed(SEED)
modele_e5 = keras.Sequential(
    [
        keras.Input(shape=IMAGE_SIZE + (1,)),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(128, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(128, activation="relu"),
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ],
    name="e5_poids_classes",
)
historique_e5 = entrainer(modele_e5, poids_classes)  # NOUVEAU : poids par classe
mesurer("e5_poids_classes", "+ class_weight (classes rares)", modele_e5, historique_e5)

# Rappel de chaque classe avant (E3) et après (E5) la pondération : qui gagne, qui perd ?
predites_e3 = modele_e3.predict(val_ds, verbose=0).argmax(axis=1)
predites_e5 = modele_e5.predict(val_ds, verbose=0).argmax(axis=1)
for numero, classe in enumerate(CLASS_NAMES):
    images_classe = vraies_classes == numero
    rappel_e3 = (predites_e3[images_classe] == numero).mean()
    rappel_e5 = (predites_e5[images_classe] == numero).mean()
    print(f"{classe:8s} rappel E3 {rappel_e3:6.1%} -> E5 {rappel_e5:6.1%}")

**Résultat E5.** Le rappel de `disgust` augmente fortement (la cellule affiche le rappel de chaque classe avec E3 et avec E5) : la pondération fait bien ce qu'on attendait. Mais l'accuracy baisse de plusieurs points et la perte de validation monte. Le détail par classe montre où sont passées ces images : `happy` perd le plus de rappel, et d'autres classes fréquentes reculent aussi. Le modèle répond plus souvent `disgust`, et une partie de ces réponses tombe sur des images d'autres classes. Comme `disgust` ne représente qu'environ 1 % des images de validation, ce qu'on gagne sur cette classe ne compense pas ce qui est perdu ailleurs.

**Décision : on ne garde pas la pondération pour le modèle final.** Le poids de `disgust` (environ 9,4) est probablement trop fort ; un poids intermédiaire (par exemple la racine carrée de ces poids) serait une piste, que nous n'avons pas testée.

### 6g — Tableau comparatif (validation)

Le tableau est affiché à partir de la liste `resultats`. « Écart » = accuracy d'entraînement moins accuracy de validation, à la meilleure époque. Avec *dropout* ou augmentation, l'accuracy d'entraînement est mesurée sur des images perturbées : l'écart y paraît donc un peu plus petit qu'il ne l'est.

In [ ]:
# Partie 6g : tableau comparatif des expériences (validation uniquement)
print(f"{'modèle':17s} {'changement':40s} {'accuracy':>8s} {'perte':>6s} {'meilleure ép.':>13s} "
      f"{'écart':>6s} {'rappel disgust':>14s}")
for ligne in resultats:
    epoque = f"{ligne['meilleure_epoque']} / {ligne['epoques']}"
    print(f"{ligne['nom']:17s} {ligne['changement']:40s} {ligne['accuracy']:8.1%} {ligne['perte']:6.3f} "
          f"{epoque:>13s} {ligne['ecart']:+6.1%} {ligne['rappel_disgust']:14.1%}")

**Résumé des expériences (validation).** Les valeurs exactes de cette exécution sont dans le tableau affiché par la cellule précédente. Ce qu'on en retient :

| Modèle | Changement | Observation | Décision |
|---|---|---|---|
| `cnn_base` | référence (Partie 3) | surapprend après quelques époques | référence |
| E1 | + `Dense(128)` cachée | accuracy presque identique, perte un peu meilleure, surapprend plus tôt | gardé |
| E2 | + `Dropout(0.5)` | perte meilleure, surapprentissage freiné | gardé |
| E3 | + 3e bloc `Conv2D(128)` | plus gros gain (plusieurs points), plus petite perte, 2,4 fois moins de paramètres que E2 | **gardé, modèle final** |
| E4 | + augmentation | ne surapprend plus, mais apprend trop lentement, en dessous de E3 | rejeté |
| E5 | E3 + `class_weight` | `disgust` bien mieux reconnu, mais `happy` et l'accuracy reculent | rejeté |

**Choix du modèle final : E3 (`e3_3_blocs`).** C'est le modèle qui a la meilleure accuracy et la plus petite perte de validation, selon les critères fixés au début de la Partie 6. Par rapport au CNN de départ, il gagne plusieurs points d'accuracy et reconnaît nettement plus d'images `disgust`. E5 reconnaît beaucoup mieux `disgust`, mais coûte plusieurs points d'accuracy : nous ne le retenons pas, car le projet compare d'abord les modèles sur l'accuracy. Si l'application devait absolument détecter le dégoût, E5 serait le meilleur choix.

Les chiffres varient un peu d'une exécution à l'autre (CPU ou GPU, version de TensorFlow). Un écart de moins d'un point entre deux modèles (comme entre `cnn_base`, E1 et E2) peut venir du hasard de l'initialisation ; les effets de E3, E4 et E5 sont assez grands pour se retrouver à chaque exécution.

### 6h — Courbes du modèle retenu

On retrace les courbes de E3 pour les comparer à celles du CNN de base (4b).

In [ ]:
# Partie 6h : courbes du modèle retenu, à comparer avec celles du CNN de base (4b)
modele_final = modele_e3
tracer_courbes(historique_e3, f"Modèle retenu ({modele_final.name}) — courbes d'apprentissage")

**Comparaison avec 4b.** Avec `cnn_base`, les courbes d'entraînement et de validation se séparaient dès les premières époques. Avec E3, elles restent ensemble plus longtemps, et la perte de validation continue de baisser lentement et descend plus bas que le meilleur score de `cnn_base`. Le surapprentissage arrive donc plus tard et plus doucement. Il n'a pas disparu : à la fin, l'accuracy d'entraînement dépasse nettement celle de validation.

### 6i — Évaluation finale sur le jeu de test

**Attention : c'est la seule fois du projet où le jeu de test est utilisé.** Toutes les décisions (architecture, *dropout*, augmentation, pondération, choix du modèle final) ont été prises avant, sur la validation. Si on se servait du test pour choisir entre plusieurs modèles, son score ne serait plus une estimation honnête de ce que le modèle fait sur des images jamais vues. Cette cellule ne doit donc pas servir à revenir modifier le modèle.

On mesure l'accuracy, la matrice de confusion et le rappel par classe, puis on sauvegarde le modèle dans `models/modele_final.keras` (dossier non versionné) pour la démonstration.

In [ ]:
# Partie 6i : évaluation finale, SEULE utilisation du jeu de test
perte_test, accuracy_test = modele_final.evaluate(test_ds, verbose=0)

# test_ds n'est pas mélangé (shuffle=False en 1c) : étiquettes et prédictions sont dans le même ordre
vraies_test = np.concatenate([labels.numpy() for images, labels in test_ds]).argmax(axis=1)
predites_test = modele_final.predict(test_ds, verbose=0).argmax(axis=1)

matrice_test = confusion_matrix(vraies_test, predites_test)
ConfusionMatrixDisplay(matrice_test, display_labels=CLASS_NAMES).plot(cmap="Blues", xticks_rotation=45)
plt.title(f"Modèle final ({modele_final.name}) — matrice de confusion (test)")
plt.show()

perte_val_final, accuracy_val_final = modele_final.evaluate(val_ds, verbose=0)
part_happy_test = (vraies_test == CLASS_NAMES.index("happy")).mean()

print(f"Modèle final : {modele_final.name}")
print(f"Accuracy sur le test : {accuracy_test:.1%}  (perte {perte_test:.3f}, {len(vraies_test)} images)")
print(f"Accuracy sur la validation (même modèle) : {accuracy_val_final:.1%}")
print(f"Repère « toujours happy » sur le test : {part_happy_test:.1%}")
for numero, classe in enumerate(CLASS_NAMES):
    total_classe = matrice_test[numero].sum()
    print(f"{classe:10s} rappel = {matrice_test[numero, numero] / total_classe:.1%}  "
          f"({matrice_test[numero, numero]} / {total_classe})")

# Sauvegarde : architecture + poids dans un seul fichier, rechargé par la démonstration
Path("models").mkdir(exist_ok=True)
modele_final.save("models/modele_final.keras")
print("Modèle sauvegardé dans models/modele_final.keras")

**Résultat sur le test.** Les scores du modèle final sur le test et sur la validation sont affichés par la cellule précédente. Ils sont proches : le choix fait sur la validation tient sur des images qui n'ont servi à aucune décision. Le petit écart entre les deux vient du simple fait que ce ne sont pas les mêmes images. Le score est bien au-dessus du repère « toujours `happy` ».

Les classes se rangent presque dans le même ordre qu'en validation : `happy` et `surprise` sont les mieux reconnues, puis `neutral`, `angry` et `sad` ; `fear` et `disgust` restent les plus difficiles. Les grandes confusions sont aussi les mêmes, autour de `sad` et `neutral`, et `disgust` est surtout pris pour `angry`.

### Démonstration : prédire l'expression d'une image

Cette cellule recharge le modèle sauvegardé et prédit l'expression d'une seule image. Pour essayer une autre image, il suffit de changer `CHEMIN_IMAGE`.

L'image est préparée **exactement comme pendant l'entraînement** : niveaux de gris, 48 × 48 pixels, valeurs divisées par 255. Sans la division par 255, le modèle recevrait des valeurs 255 fois trop grandes et ses prédictions n'auraient plus de sens. Le modèle a appris sur des visages déjà recadrés : sur une photo où le visage est petit ou décentré, il se trompera (c'est le rôle de la détection de visages des parties 8 et 9).

In [ ]:
# Démonstration : une image -> les 7 probabilités et la classe prédite
CHEMIN_IMAGE = sorted((DATA_DIR / "test" / "happy").glob("*.jpg"))[0]  # à remplacer par une autre image

modele_demo = keras.models.load_model("models/modele_final.keras")

# Même préparation qu'à l'entraînement : niveaux de gris, 48x48, pixels entre 0 et 1
image = keras.utils.load_img(CHEMIN_IMAGE, color_mode="grayscale", target_size=IMAGE_SIZE)
pixels = keras.utils.img_to_array(image) / 255.0  # forme (48, 48, 1)
lot = np.expand_dims(pixels, axis=0)              # forme (1, 48, 48, 1) : un lot d'une seule image
probabilites_demo = modele_demo.predict(lot, verbose=0)[0]
classe_demo = CLASS_NAMES[int(probabilites_demo.argmax())]

fig, (ax_image, ax_barres) = plt.subplots(1, 2, figsize=(10, 3.5))
ax_image.imshow(pixels, cmap="gray", vmin=0, vmax=1)
ax_image.set_title(f"prédiction : {classe_demo} ({probabilites_demo.max():.0%})")
ax_image.axis("off")
ax_barres.barh(CLASS_NAMES, probabilites_demo, color="#2a78d6")
ax_barres.invert_yaxis()  # angry en haut, comme dans CLASS_NAMES
ax_barres.set_xlim(0, 1)
ax_barres.set_xlabel("probabilité")
plt.tight_layout()
plt.show()

print(f"Image : {CHEMIN_IMAGE}")
print(f"Dossier d'origine (vraie classe) : {Path(CHEMIN_IMAGE).parent.name}")
for classe, probabilite in zip(CLASS_NAMES, probabilites_demo):
    print(f"{classe:10s} {probabilite:.2f}")
print(f"Classe prédite : {classe_demo}")